WEEK 2




In [ ]:
!pip install pandas numpy

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving attendances.csv to attendances.csv


In [ ]:
df = pd.read_csv("attendances.csv")
print(df)

   employee_id employee_name department             clock_in  \
0            1         Rahul         HR  2026-05-01 09:00:00   
1            2         Sneha         IT  2026-05-01 09:30:00   
2            3          Amit    Finance  2026-05-01 10:00:00   
3            4         Priya         HR  2026-05-01 09:15:00   
4            5         Kiran         IT  2026-05-01 11:00:00   

             clock_out  tasks_completed  
0  2026-05-01 18:00:00               10  
1  2026-05-01 17:00:00                7  
2  2026-05-01 19:00:00               12  
3  2026-05-01 18:30:00                9  
4  2026-05-01 17:00:00                5  


In [ ]:
df['clock_in'] = pd.to_datetime(df['clock_in'])
df['clock_out'] = pd.to_datetime(df['clock_out'])

In [ ]:
df['work_hours'] = (
    df['clock_out'] - df['clock_in']
).dt.total_seconds() / 3600

In [ ]:
df['productivity_score'] = (
    df['tasks_completed'] / df['work_hours']
)

In [ ]:
top = df.sort_values(
    by='productivity_score',
    ascending=False
)

print(top)

   employee_id employee_name department            clock_in  \
2            3          Amit    Finance 2026-05-01 10:00:00   
0            1         Rahul         HR 2026-05-01 09:00:00   
3            4         Priya         HR 2026-05-01 09:15:00   
1            2         Sneha         IT 2026-05-01 09:30:00   
4            5         Kiran         IT 2026-05-01 11:00:00   

            clock_out  tasks_completed  work_hours  productivity_score  
2 2026-05-01 19:00:00               12        9.00            1.333333  
0 2026-05-01 18:00:00               10        9.00            1.111111  
3 2026-05-01 18:30:00                9        9.25            0.972973  
1 2026-05-01 17:00:00                7        7.50            0.933333  
4 2026-05-01 17:00:00                5        6.00            0.833333  


In [ ]:
absent = df[df['work_hours'] < 7]

print(absent)

   employee_id employee_name department            clock_in  \
4            5         Kiran         IT 2026-05-01 11:00:00   

            clock_out  tasks_completed  work_hours  productivity_score  
4 2026-05-01 17:00:00                5         6.0            0.833333  


WEEK 3

In [ ]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AttendanceTracker") \
    .getOrCreate()

In [6]:
from google.colab import files
uploaded = files.upload()

Saving attendances.csv to attendances.csv


In [7]:
df = spark.read.csv(
    "attendances.csv",
    header=True,
    inferSchema=True
)

In [16]:
df.show()

+-----------+-------------+----------+-------------------+-------------------+---------------+
|employee_id|employee_name|department|           clock_in|          clock_out|tasks_completed|
+-----------+-------------+----------+-------------------+-------------------+---------------+
|          1|        Rahul|        HR|2026-05-01 09:00:00|2026-05-01 18:00:00|             10|
|          2|        Sneha|        IT|2026-05-01 09:30:00|2026-05-01 17:00:00|              7|
|          3|         Amit|   Finance|2026-05-01 10:00:00|2026-05-01 19:00:00|             12|
|          4|        Priya|        HR|2026-05-01 09:15:00|2026-05-01 18:30:00|              9|
|          5|        Kiran|        IT|2026-05-01 11:00:00|2026-05-01 17:00:00|              5|
+-----------+-------------+----------+-------------------+-------------------+---------------+



In [17]:
df.filter(df.clock_in > '09:00:00').show()


+-----------+-------------+----------+--------+---------+---------------+
|employee_id|employee_name|department|clock_in|clock_out|tasks_completed|
+-----------+-------------+----------+--------+---------+---------------+
+-----------+-------------+----------+--------+---------+---------------+



In [18]:
from pyspark.sql.functions import col
df = df.withColumn(
    "work_hours",
    (
        col("clock_out").cast("long") -
        col("clock_in").cast("long")
    ) / 3600
)

In [19]:
df.show()

+-----------+-------------+----------+-------------------+-------------------+---------------+----------+
|employee_id|employee_name|department|           clock_in|          clock_out|tasks_completed|work_hours|
+-----------+-------------+----------+-------------------+-------------------+---------------+----------+
|          1|        Rahul|        HR|2026-05-01 09:00:00|2026-05-01 18:00:00|             10|       9.0|
|          2|        Sneha|        IT|2026-05-01 09:30:00|2026-05-01 17:00:00|              7|       7.5|
|          3|         Amit|   Finance|2026-05-01 10:00:00|2026-05-01 19:00:00|             12|       9.0|
|          4|        Priya|        HR|2026-05-01 09:15:00|2026-05-01 18:30:00|              9|      9.25|
|          5|        Kiran|        IT|2026-05-01 11:00:00|2026-05-01 17:00:00|              5|       6.0|
+-----------+-------------+----------+-------------------+-------------------+---------------+----------+



In [20]:
from pyspark.sql.functions import avg , col

df.groupBy("department") \
  .agg(avg("work_hours")) \
  .show()

+----------+---------------+
|department|avg(work_hours)|
+----------+---------------+
|        HR|          9.125|
|   Finance|            9.0|
|        IT|           6.75|
+----------+---------------+



In [21]:
issues = df.filter(df.work_hours < 7)

issues.show()

+-----------+-------------+----------+-------------------+-------------------+---------------+----------+
|employee_id|employee_name|department|           clock_in|          clock_out|tasks_completed|work_hours|
+-----------+-------------+----------+-------------------+-------------------+---------------+----------+
|          5|        Kiran|        IT|2026-05-01 11:00:00|2026-05-01 17:00:00|              5|       6.0|
+-----------+-------------+----------+-------------------+-------------------+---------------+----------+



week 4

In [22]:
df = df.withColumn(
    "productivity_score",
    col("tasks_completed") / col("work_hours")
)

In [23]:
from pyspark.sql.functions import avg

kpi = df.groupBy("department") \
        .agg(avg("productivity_score"))

kpi.show()

+----------+-----------------------+
|department|avg(productivity_score)|
+----------+-----------------------+
|        HR|     1.0420420420420422|
|   Finance|     1.3333333333333333|
|        IT|     0.8833333333333333|
+----------+-----------------------+



In [24]:
kpi.write.csv(
    "/FileStore/output",
    header=True
)